# Apache Superset

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/superset/superset.ipynb)

Apache Superset is an open-source BI platform: a Flask web app that stores only metadata (connections, datasets, charts, dashboards, permissions) and sends SQL to your databases through SQLAlchemy. This notebook covers datasets and the semantic layer, how a chart becomes SQL, Jinja templating, caching and async queries, row-level security, embedding, dashboards as code and performance. Superset itself needs a server, so the mechanics are reproduced with DuckDB and Jinja.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0", "pandas": "pandas>=2.2", "jinja2": "jinja2>=3.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## Architecture

| Component | Role |
|---|---|
| Web server (Gunicorn + Flask) | UI, REST API, chart query execution for synchronous queries |
| Metadata database (PostgreSQL or MySQL) | Users, roles, connections, datasets, charts (`slices`), dashboards, RLS rules, query history. SQLite is for demos only. |
| Cache (Redis) | Chart data results, dashboard metadata, thumbnails, async query results |
| Celery workers + beat | Async SQL Lab queries, alerts and reports, thumbnails, cache warm-up; beat schedules them |
| Headless browser (Playwright/Selenium) | Screenshots for reports and thumbnails |
| Databases | Anything with a SQLAlchemy dialect and a Superset *engine spec* (Postgres, Trino, BigQuery, Snowflake, ClickHouse, Druid, Databricks, DuckDB, ...) |

The engine spec describes each database's quirks: time-grain SQL expressions, how to apply limits, how to cancel queries, whether it supports subqueries and joins. Install the driver in the Superset image (`pip install trino`) and add a connection with a SQLAlchemy URI such as `trino://user@trino.internal:443/hive`; extra JSON holds engine parameters, and options such as "Allow DML", "Expose in SQL Lab", "Allow file uploads" and per-database cache timeout are set on the connection.

Installation paths: Docker Compose for local trials, the official Helm chart for Kubernetes in production, or `pip install apache-superset` followed by `superset db upgrade`, `superset fab create-admin` and `superset init`. In production run several stateless web pods behind a load balancer, a separate Celery worker deployment (scaled independently), one beat process, managed PostgreSQL and Redis.

Everything is configured in `superset_config.py`, loaded from `SUPERSET_CONFIG_PATH`:

```python
# superset_config.py
import os

SECRET_KEY = os.environ["SUPERSET_SECRET_KEY"]                 # encrypts DB passwords in the metadata DB
SQLALCHEMY_DATABASE_URI = os.environ["SUPERSET_METADATA_URI"]  # postgresql+psycopg2://...
ROW_LIMIT = 10_000
SQLLAB_TIMEOUT = 300
SUPERSET_WEBSERVER_TIMEOUT = 120

DATA_CACHE_CONFIG = {"CACHE_TYPE": "RedisCache", "CACHE_DEFAULT_TIMEOUT": 3600,
                     "CACHE_REDIS_URL": os.environ["REDIS_URL"]}
CACHE_CONFIG = {"CACHE_TYPE": "RedisCache", "CACHE_DEFAULT_TIMEOUT": 300,
                "CACHE_REDIS_URL": os.environ["REDIS_URL"]}

FEATURE_FLAGS = {"ALERT_REPORTS": True, "EMBEDDED_SUPERSET": True,
                 "GLOBAL_ASYNC_QUERIES": True, "DASHBOARD_RBAC": True}

class CeleryConfig:
    broker_url = os.environ["REDIS_URL"]
    result_backend = os.environ["REDIS_URL"]
    imports = ("superset.sql_lab", "superset.tasks.scheduler")

CELERY_CONFIG = CeleryConfig
```

Superset compared with other tools: it is free and open source with a strong SQL Lab and many chart types, but its semantic layer is thin (metrics live per dataset, there is no modeled join graph like LookML), and pixel-perfect reports or heavy governance need extra work. Many teams pair it with dbt (and sometimes a semantic layer such as Cube) so that business logic lives upstream.

## Datasets: the semantic layer

A **dataset** is the unit charts query. It is either:

+ **Physical**: points at one table or view. Superset can push filters and aggregates straight into it; easiest to optimize.
+ **Virtual**: defined by a SQL query written in SQL Lab. Superset wraps it as a subquery on every chart query, so its cost is paid every time.

On a dataset you define **metrics** (`SUM(net_revenue)`, `COUNT(DISTINCT customer_id)`), **calculated columns** (`CASE WHEN ... END`), the **main datetime column** and default time grain, column flags (groupable, filterable, is temporal), certification and a cache timeout. Defining metrics once on the dataset is what keeps "revenue" consistent; ad-hoc metrics written inside individual charts drift.

Modeling guidance: point Superset at a dbt-built star or wide table at the dashboard grain; prefer physical datasets over views over virtual datasets; avoid one virtual dataset per chart; keep joins out of Superset where possible (there is no join modeling, so every virtual dataset re-implements them).

The next cell shows the query shape Superset generates for a "SUM(net_revenue) by week and country" chart on a virtual dataset: the dataset SQL becomes `virtual_table`, the time grain becomes a dialect-specific expression, and the row limit is appended.

### Where you'll use it

- **Self-hosted BI:** a company that wants open-source dashboards on its own warehouse runs Superset, with metadata in its own database.

### Where you'll use it

- **Shared definitions:** a dataset defines the metric columns once, so every chart built from it uses the same logic.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("""
CREATE TABLE fact_orders AS
SELECT
    i AS order_id,
    TIMESTAMP '2026-01-01' + (i % 90) * INTERVAL 1 DAY + (i % 24) * INTERVAL 1 HOUR AS order_ts,
    ['US', 'DE', 'FR', 'IN'][1 + i % 4] AS country_code,
    ['m1', 'm2', 'm3'][1 + (i // 4) % 3] AS merchant_id,
    'c' || (i % 700) AS customer_id,
    ROUND(5 + (i * 13) % 200, 2) AS net_revenue,
    i % 17 = 0 AS is_test
FROM range(20000) r(i)
""")

VIRTUAL_DATASET_SQL = "SELECT * FROM fact_orders WHERE NOT is_test"
TIME_GRAINS = {  # what DuckDB's engine spec would map each grain to
    "PT1H": "DATE_TRUNC('hour', {col})",
    "P1D": "DATE_TRUNC('day', {col})",
    "P1W": "DATE_TRUNC('week', {col})",
    "P1M": "DATE_TRUNC('month', {col})",
}


def chart_query(dataset_sql: str, time_col: str, grain: str, groupby: list[str],
                metrics: dict[str, str], row_limit: int = 10_000) -> str:
    time_expr = TIME_GRAINS[grain].format(col=time_col)
    select = [f"{time_expr} AS __timestamp", *groupby] + [f"{sql} AS {name}" for name, sql in metrics.items()]
    group_positions = ", ".join(str(i) for i in range(1, len(groupby) + 2))
    return (f"SELECT {', '.join(select)}\nFROM ({dataset_sql}) AS virtual_table\n"
            f"GROUP BY {group_positions}\nORDER BY 1\nLIMIT {row_limit}")


sql = chart_query(VIRTUAL_DATASET_SQL, "order_ts", "P1W", ["country_code"], {"revenue": "SUM(net_revenue)"})
print(sql)
weekly = con.sql(sql).df()
print(weekly.head())
assert weekly["__timestamp"].nunique() <= 14 and set(weekly["country_code"]) == {"US", "DE", "FR", "IN"}

## Charts, Explore and dashboards

+ **Explore** is the chart builder: pick a dataset, a visualization type, dimensions, metrics, filters, time range and grain; "View query" shows the generated SQL. A saved chart (historically a *slice*) stores its form data as JSON in the metadata DB.
+ **Visualization types** cover tables, pivot tables, time-series line/bar/area, big number with trendline, pie, heatmap, treemap, sunburst, box plot, maps (deck.gl), Sankey and more, mostly built on Apache ECharts. Custom visualizations are React plugins built with `@superset-ui` and bundled into a custom frontend build.
+ **Dashboards** arrange charts in a grid with tabs, rows, markdown and dividers. **Native filters** (filter bar) target selected charts and datasets; filters can be dependent (country restricts city values), and **cross-filtering** turns a click on a chart into a filter. Drill to detail and drill by give row-level and per-dimension exploration.
+ Charts from several databases can live on one dashboard, but each chart reads one dataset; Superset does not join across databases (use Trino or the warehouse for that).

Every chart is one database query at render time (plus one per filter needing value lists), unless served from cache:

In [ ]:
def dashboard_queries(charts: int, select_filters: int, cache_hit_rate: float) -> float:
    filter_value_queries = select_filters  # each select filter fetches its distinct values
    return (charts + filter_value_queries) * (1 - cache_hit_rate)


cold = dashboard_queries(charts=30, select_filters=5, cache_hit_rate=0.0)
warm = dashboard_queries(charts=30, select_filters=5, cache_hit_rate=0.9)
print(f"one cold render: {cold:.0f} queries; with 90% cache hits: {warm:.1f}")
print(f"200 viewers at 9am, cold cache: {cold * 200:,.0f} database queries")
assert cold == 35 and round(warm, 1) == 3.5

## SQL Lab and Jinja templating

**SQL Lab** is the SQL IDE: schema browser, query history, saved queries, CSV export, and "Save as dataset" to turn a query into a virtual dataset. Long queries run asynchronously on Celery workers when the database is configured for async execution; results are stored in a results backend (S3 or Redis) and fetched by the browser. Access is controlled per database (`database_access`) and schema (`schema_access`); DML is disabled unless "Allow DML" is on.

With `ENABLE_TEMPLATE_PROCESSING` enabled, SQL Lab queries and virtual datasets are **Jinja templates**. Built-in macros include `{{ current_username() }}`, `{{ current_user_id() }}`, `{{ url_param('x') }}`, `{{ filter_values('col') }}`, `{{ get_filters('col') }}`, `{{ from_dttm }}` / `{{ to_dttm }}` and `{{ dataset(42) }}`. `JINJA_CONTEXT_ADDONS` in the config adds your own macros. The main use is pushing a dashboard filter *inside* a virtual dataset's CTE so the database can prune partitions before an expensive aggregation:

In [ ]:
from jinja2 import Environment

TEMPLATE = """
WITH recent AS (
    SELECT * FROM fact_orders
    WHERE order_ts >= '{{ from_dttm }}' AND order_ts < '{{ to_dttm }}'
    {% if filter_values('country_code') %}
      AND country_code IN ({{ filter_values('country_code') | map('tojson') | join(', ') | replace('"', "'") }})
    {% endif %}
)
SELECT country_code, COUNT(*) AS orders, SUM(net_revenue) AS revenue
FROM recent GROUP BY 1 ORDER BY 1
"""


def render(template: str, native_filters: dict[str, list[str]], from_dttm: str, to_dttm: str) -> str:
    """Render like Superset: dashboard filter values and time range become template context."""
    env = Environment(autoescape=False)
    return env.from_string(template).render(
        filter_values=lambda column: native_filters.get(column, []),
        from_dttm=from_dttm,
        to_dttm=to_dttm,
    )


filtered = render(TEMPLATE, {"country_code": ["DE", "FR"]}, "2026-02-01", "2026-03-01")
print(filtered)
result = con.sql(filtered).df()
print(result)
assert result["country_code"].tolist() == ["DE", "FR"]

unfiltered = con.sql(render(TEMPLATE, {}, "2026-02-01", "2026-03-01")).df()
assert len(unfiltered) == 4

Template safety: Jinja output is pasted into SQL, so a user-controlled `url_param` or filter value is an injection risk. Quote values (Superset provides `| where_in`), keep templating out of datasets editable by untrusted users, and remember that templated virtual datasets cannot be cached as effectively because each filter combination is a different query.

## Caching and async queries

Superset caches the *result* of each chart query in `DATA_CACHE_CONFIG`. The cache key is a hash of the query object: dataset, columns, metrics, filters, time range, row limit, RLS clauses and the rendered SQL. Two users with different RLS rules therefore never share entries. The timeout resolves from the most specific setting: chart, then dataset, then database, then `CACHE_DEFAULT_TIMEOUT`.

+ Relative time ranges ("Last week") are resolved at query time, so the key changes as time passes.
+ "Force refresh" on a dashboard bypasses the cache.
+ Cache warm-up tasks (Celery beat) can pre-run top dashboards right after the ETL load so the 9am wave hits a warm cache.
+ `GLOBAL_ASYNC_QUERIES` moves chart queries to Celery workers, so slow queries do not tie up web workers and time out at `SUPERSET_WEBSERVER_TIMEOUT`.

### Where you'll use it

- **Large dashboards:** async queries keep a slow report from blocking the page, and results are cached for repeat viewers.

In [ ]:
import hashlib
import json

DEFAULT_TIMEOUT = 86_400


def cache_key(query_object: dict) -> str:
    return hashlib.md5(json.dumps(query_object, sort_keys=True).encode()).hexdigest()


def resolve_timeout(chart: int | None, dataset: int | None, database: int | None) -> int:
    return next(t for t in (chart, dataset, database, DEFAULT_TIMEOUT) if t is not None)


base = {"dataset": 17, "metrics": ["SUM(net_revenue)"], "groupby": ["country_code"],
        "time_range": "2026-02-01 : 2026-03-01"}
same_for_everyone = cache_key(base) == cache_key(dict(base))
tenant_a = cache_key({**base, "rls": ["merchant_id = 'm1'"]})
tenant_b = cache_key({**base, "rls": ["merchant_id = 'm2'"]})
print("identical queries share a key:", same_for_everyone)
print("RLS splits the key:", tenant_a != tenant_b)
assert same_for_everyone and tenant_a != tenant_b
assert resolve_timeout(None, 600, 3600) == 600
assert resolve_timeout(None, None, None) == DEFAULT_TIMEOUT

Sizing workers for async queries is queueing arithmetic: concurrent slow queries times their duration must fit in worker slots, otherwise queries wait in the queue.

In [ ]:
import math

PEAK_DASHBOARD_LOADS_PER_MIN = 120
CHARTS_PER_DASHBOARD = 12
CACHE_HIT_RATE = 0.8
AVG_QUERY_SECONDS = 3.0
CONCURRENCY_PER_WORKER = 8  # celery --concurrency


queries_per_second = PEAK_DASHBOARD_LOADS_PER_MIN * CHARTS_PER_DASHBOARD * (1 - CACHE_HIT_RATE) / 60
busy_slots = queries_per_second * AVG_QUERY_SECONDS  # Little's law: L = lambda * W
workers = math.ceil(busy_slots / CONCURRENCY_PER_WORKER * 1.5)  # 50% headroom
print(f"{queries_per_second:.1f} q/s -> {busy_slots:.1f} busy slots -> {workers} worker pods")
assert round(busy_slots, 1) == 14.4 and workers == 3

## Security: roles, permissions and row-level security

Superset's security model comes from Flask-AppBuilder. Built-in roles: **Admin**, **Alpha** (all data, can create content), **Gamma** (only the data sources granted to them, can create charts on those), **sql_lab** (SQL Lab access), **Public**. Best practice is to leave built-in roles untouched and grant data through additional roles carrying `datasource_access`, `schema_access` or `database_access` permissions. `DASHBOARD_RBAC` restricts dashboards to roles. Authentication: database users, LDAP, OAuth/OIDC (via `AUTH_TYPE` and a custom security manager), or a reverse proxy header.

**Row-level security** rules attach a SQL `WHERE` clause to datasets for roles:

+ **Regular** filters apply to the listed roles.
+ **Base** filters apply to everyone *except* the listed roles (for example "everyone sees only non-test rows, except Admin").
+ Rules with the same **group key** are combined with `OR`; different group keys are combined with `AND`. Clauses support Jinja (`merchant_id = '{{ current_username() }}'`).

The combination logic is the part people get wrong, so here it is executed:

### Where you'll use it

- **Multi-team analytics:** row-level security filters a shared dashboard so each sales region sees only its own rows.

In [ ]:
RLS_RULES = [
    {"type": "Regular", "roles": {"emea_sales"}, "group_key": "geo", "clause": "country_code = 'DE'"},
    {"type": "Regular", "roles": {"emea_sales"}, "group_key": "geo", "clause": "country_code = 'FR'"},
    {"type": "Regular", "roles": {"merchant_m1"}, "group_key": "merchant", "clause": "merchant_id = 'm1'"},
    {"type": "Base", "roles": {"Admin"}, "group_key": None, "clause": "NOT is_test"},
]


def rls_where(user_roles: set[str]) -> str:
    groups: dict[str, list[str]] = {}
    for index, rule in enumerate(RLS_RULES):
        applies = bool(user_roles & rule["roles"]) if rule["type"] == "Regular" else not user_roles & rule["roles"]
        if applies:
            groups.setdefault(rule["group_key"] or f"_ungrouped_{index}", []).append(rule["clause"])
    clauses = ["(" + " OR ".join(group) + ")" for group in groups.values()]
    return " AND ".join(clauses) if clauses else "TRUE"


for roles in [{"emea_sales"}, {"emea_sales", "merchant_m1"}, {"Admin"}]:
    where = rls_where(roles)
    rows = con.sql(f"SELECT COUNT(*) FROM fact_orders WHERE {where}").fetchall()[0][0]
    print(f"{sorted(roles)!s:32} rows={rows:6d}  WHERE {where}")

assert rls_where({"Admin"}) == "TRUE"
assert rls_where({"emea_sales"}) == "(country_code = 'DE' OR country_code = 'FR') AND (NOT is_test)"
countries = con.sql(f"SELECT DISTINCT country_code FROM fact_orders WHERE {rls_where({'emea_sales'})}").df()
assert set(countries["country_code"]) == {"DE", "FR"}

Superset has no column masking engine: restrict columns by exposing curated datasets or views per audience, or rely on warehouse masking policies with per-user credentials (impersonation is supported for some engines such as Trino and Hive). Audit: the `logs` table records actions (dashboard views, chart queries, exports) and the `query` table stores SQL Lab history; ship both to your log platform, or enable `EVENT_LOGGER` to a custom sink.

## Embedding and multi-tenancy

The embedded SDK (`@superset-ui/embedded-sdk`) renders a dashboard in an iframe authenticated by a **guest token** that your backend requests with a service account holding `can_grant_guest_token`. The token names the dashboards and carries RLS clauses, so tenant scoping is decided server-side:

```python
payload = {
    "user": {"username": "merchant_4821", "first_name": "Merchant", "last_name": "4821"},
    "resources": [{"type": "dashboard", "id": "<embedded-dashboard-uuid>"}],
    "rls": [{"clause": "merchant_id = '4821'"}],
}
token = session.post(f"{SUPERSET_URL}/api/v1/security/guest_token/", json=payload).json()["token"]
```

Multi-tenancy options: one dataset with RLS by tenant (cheapest to maintain, cache split per tenant); per-tenant schemas or databases with per-tenant datasets (strong isolation, many objects to manage, good fit for strict contractual separation); or separate Superset deployments for regulated tenants.

## Reports, alerts and orchestration

With `ALERT_REPORTS`, Celery beat schedules **reports** (dashboard or chart screenshots, CSV, to email or Slack) and **alerts** (a SQL query whose result is compared to a threshold; notify when it fires). They need workers, beat, and a headless browser in the worker image.

Superset has no built-in Airflow integration, but they combine well: Airflow loads and tests data, then calls the Superset REST API to refresh dataset metadata, warm caches for key dashboards or trigger reports, so dashboards never show half-loaded data. Superset also cannot "stream": near-real-time dashboards are a matter of a fast database (Druid, Pinot, ClickHouse), short cache timeouts and dashboard auto-refresh intervals.

## Dashboards as code

Charts, datasets, databases and dashboards export as a ZIP of YAML files (UI, `superset export-dashboards`, or `GET /api/v1/dashboard/export/`) and import into another environment (`POST /api/v1/dashboard/import/` with `overwrite`). Store the YAML in Git, review diffs, and promote dev -> staging -> prod in CI. Database passwords are not exported; supply them at import time. Tools such as `preset-cli` (`sup`) add templating per environment.

```bash
superset export-dashboards -f dashboards.zip
unzip dashboards.zip -d assets/ && git add assets/
superset import-dashboards -p dashboards.zip -u admin
```

## Performance

### Diagnosing a slow chart

1. Open "View query" and run it in SQL Lab or the database console. If the SQL itself is slow, it is a modeling problem.
2. Check whether it was a cache miss (chart header shows cached time; `force=true` reproduces a miss).
3. Check queueing: web worker timeouts (`SUPERSET_WEBSERVER_TIMEOUT`), Celery queue depth, database connection pool limits.
4. Check rendering: very large results (a raised `ROW_LIMIT`) cost browser time.

### Reducing database load

+ Pre-aggregate to dashboard grain in dbt; a 500M-row fact behind a daily-by-country chart should be a few thousand rows.
+ Prefer physical datasets; push filters into virtual datasets with Jinja; always set a default time range.
+ Fewer charts per dashboard; use tabs (charts in hidden tabs load lazily).
+ Cache with timeouts matched to load frequency and warm caches after loads.
+ Async queries for anything slower than a few seconds; size workers and database pools together.
+ Monitor with StatsD metrics (`STATS_LOGGER`), Gunicorn and Celery metrics, slow query logs in the database, and the `logs` table for dashboard popularity.

### Where you'll use it

- **Customer-facing analytics:** a SaaS product embeds a dashboard that shows each customer only their own data.

### Where you'll use it

- **Scheduled reports:** a weekly PDF of KPIs is sent to leadership, and an alert fires when a metric drops.

In [ ]:
con.execute("""
CREATE TABLE agg_orders_daily AS
SELECT DATE_TRUNC('day', order_ts) AS order_date, country_code, merchant_id,
       SUM(net_revenue) AS net_revenue, COUNT(*) AS orders
FROM fact_orders WHERE NOT is_test GROUP BY 1, 2, 3
""")
fact_rows = con.sql("SELECT COUNT(*) FROM fact_orders").fetchall()[0][0]
agg_rows = con.sql("SELECT COUNT(*) FROM agg_orders_daily").fetchall()[0][0]

from_fact = con.sql("SELECT country_code, SUM(net_revenue) FROM fact_orders WHERE NOT is_test GROUP BY 1 ORDER BY 1").fetchall()
from_agg = con.sql("SELECT country_code, SUM(net_revenue) FROM agg_orders_daily GROUP BY 1 ORDER BY 1").fetchall()
print(f"fact rows={fact_rows:,} aggregate rows={agg_rows:,} ({fact_rows / agg_rows:.0f}x fewer)")
assert [(c, round(v, 2)) for c, v in from_fact] == [(c, round(v, 2)) for c, v in from_agg]

## Practice scenarios

+ **Retail self-service**: dbt star schema, physical datasets with certified metrics, cache warm-up after the nightly load.
+ **Fintech embedded analytics**: one dataset with merchant RLS, guest tokens minted per request, a test that every embedded dataset has a tenant rule.
+ **SaaS product analytics on ClickHouse**: event tables with materialized views for rollups, async queries, short cache timeouts for near-real-time charts.
+ **Platform team**: dashboards exported to Git and promoted through CI, with database secrets injected at import.

Questions to check yourself:

1. Why are virtual datasets slower than physical ones, and how does Jinja help?
2. Two RLS rules share a group key and a third has a different one: how are they combined?
3. Why do RLS users see lower cache hit rates?
4. What breaks if Celery beat is not running?